# 📓 Chapter 1 — Practice Notebook
## An Introduction to Large Language Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

### How to use this notebook

- Run cells top to bottom using `Shift + Enter`
- Every `# 🏋️ EXERCISE` cell is something **you** complete — try before looking at the solution
- Every `# ✅ SOLUTION` cell has the answer — only run it after you've tried
- Don't just execute cells blindly — read every explanation and make sure it clicks

### What you will practice
1. Bag-of-Words — build it from scratch, find its limits
2. Word Embeddings — explore real semantic similarity
3. Text Generation — run a real LLM and generate your first text
4. Context Window — see multi-turn conversation in action
5. Knowledge Check — answer questions from memory

---

## 🛠️ Setup
Run this first. Installs everything needed for this notebook.

In [ ]:
# Install required libraries
# On Google Colab this takes about 1-2 minutes
!pip install transformers accelerate sentence-transformers scikit-learn -q

print("✅ All packages installed successfully!")

---
## Part 1 — Bag-of-Words

### Why start here?
Before understanding why modern LLMs are powerful, you need to understand what came before them and why it wasn't enough. Bag-of-Words is the baseline — we build it from scratch so you truly understand its limitations.

**Core idea:** Convert text into numbers by counting word occurrences.

In [ ]:
# ── DEMO: Build Bag-of-Words from scratch ────────────────────────────────────
from collections import Counter

sentences = [
    "I love machine learning",
    "I love deep learning",
    "machine learning is fun"
]

# Step 1: Build vocabulary — all unique words across all sentences
all_words = [word.lower() for sentence in sentences for word in sentence.split()]
vocab = sorted(set(all_words))
print("Vocabulary:", vocab)
print(f"Vocabulary size: {len(vocab)} words")
print()

# Step 2: Convert each sentence to a vector (count occurrences)
def sentence_to_bow(sentence, vocab):
    word_count = Counter(sentence.lower().split())
    return [word_count.get(word, 0) for word in vocab]

print(f"{'Sentence':<35} {'Vector'}")
print("-" * 65)
for sentence in sentences:
    vector = sentence_to_bow(sentence, vocab)
    print(f"{sentence:<35} {vector}")

In [ ]:
# 🏋️ EXERCISE 1 — The Word Order Problem
# ─────────────────────────────────────────────────────────────────────────────
# These two sentences mean COMPLETELY opposite things:
#   A: "The dog bit the man"
#   B: "The man bit the dog"
#
# Tasks:
#   1. Create BoW vectors for both sentences
#   2. Check if the vectors are identical
#   3. Print what this tells us about the limitation of Bag-of-Words
#
# Hint: use sentence_to_bow() defined above

sentence_a = "The dog bit the man"
sentence_b = "The man bit the dog"

# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION — only run after attempting above

vocab_ab = sorted(set(sentence_a.lower().split() + sentence_b.lower().split()))
vec_a = sentence_to_bow(sentence_a, vocab_ab)
vec_b = sentence_to_bow(sentence_b, vocab_ab)

print("Vocabulary:", vocab_ab)
print()
print(f"Sentence A: '{sentence_a}'")
print(f"Vector A:    {vec_a}")
print()
print(f"Sentence B: '{sentence_b}'")
print(f"Vector B:    {vec_b}")
print()
print(f"Are vectors identical? {vec_a == vec_b}")
print()
print("💡 KEY INSIGHT:")
print("   Two sentences with OPPOSITE meanings produce IDENTICAL vectors.")
print("   BoW throws away word order entirely — it's literally just a bag.")
print("   This is why we needed smarter representations like word2vec.")

In [ ]:
# 🏋️ EXERCISE 2 — The Synonym Problem
# ─────────────────────────────────────────────────────────────────────────────
# BoW also treats synonyms as completely unrelated.
#
# Tasks:
#   1. Create BoW vectors for these two sentences:
#      A: "I drive a car to work"
#      B: "I drive an automobile to work"
#   2. Check if they are identical
#   3. What does this tell you?

# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION

s1 = "I drive a car to work"
s2 = "I drive an automobile to work"

combined_vocab = sorted(set(s1.lower().split() + s2.lower().split()))
v1 = sentence_to_bow(s1, combined_vocab)
v2 = sentence_to_bow(s2, combined_vocab)

print(f"'{s1}'\n  Vector: {v1}")
print()
print(f"'{s2}'\n  Vector: {v2}")
print()
print(f"Are they identical? {v1 == v2}")
print()
print("💡 KEY INSIGHT:")
print("   'car' and 'automobile' mean the same thing but BoW treats them")
print("   as completely different. It has no concept of meaning — only exact")
print("   string matching. This is why we needed embeddings.")

---
## Part 2 — Word Embeddings

### Why this matters
Embeddings solve both problems BoW had — they capture **meaning** and handle synonyms naturally. Words with similar meanings end up with similar vectors. We can even measure *how similar* two pieces of text are using cosine similarity.

**Cosine similarity:** A score from 0.0 to 1.0 — higher means more similar in meaning.

In [ ]:
# ── DEMO: Load a real embedding model ────────────────────────────────────────
# We use all-MiniLM-L6-v2 — a small, fast, high-quality sentence embedding model
# This is a REPRESENTATION model (encoder-only) — exactly what the book describes

from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading embedding model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded!")

# Embed a few words
sample_words = ["king", "queen", "man", "woman", "apple", "banana", "car", "truck"]
embeddings = embed_model.encode(sample_words)

print(f"\nEach word/phrase is represented as a vector of {embeddings.shape[1]} numbers")
print(f"\nExample — 'king' embedding (first 8 values out of {embeddings.shape[1]}):")
print(np.round(embeddings[0][:8], 3))

In [ ]:
# ── DEMO: Semantic Similarity ─────────────────────────────────────────────────
# Words with similar meanings should score HIGH
# Unrelated words should score LOW

from sklearn.metrics.pairwise import cosine_similarity

def get_similarity(word1, word2, word_list, emb_matrix):
    idx1 = word_list.index(word1)
    idx2 = word_list.index(word2)
    score = cosine_similarity([emb_matrix[idx1]], [emb_matrix[idx2]])[0][0]
    return round(float(score), 3)

pairs = [
    ("king",   "queen"),   # both royalty → should be HIGH
    ("man",    "woman"),   # both human   → should be HIGH
    ("apple",  "banana"),  # both fruits  → should be MEDIUM-HIGH
    ("car",    "truck"),   # both vehicles→ should be HIGH
    ("king",   "apple"),   # unrelated    → should be LOW
    ("queen",  "truck"),   # unrelated    → should be LOW
]

print("Semantic Similarity Scores (0.0 = unrelated, 1.0 = identical meaning)")
print("=" * 55)
for w1, w2 in pairs:
    score = get_similarity(w1, w2, sample_words, embeddings)
    bar = "█" * int(score * 30)
    print(f"{w1:8} ↔ {w2:8}  {score:.3f}  {bar}")

In [ ]:
# 🏋️ EXERCISE 3 — Synonyms vs BoW
# ─────────────────────────────────────────────────────────────────────────────
# Earlier, BoW gave us DIFFERENT vectors for 'car' and 'automobile'.
# Let's see what EMBEDDINGS give us for the same pair.
#
# Tasks:
#   1. Encode the words "car" and "automobile" using embed_model
#   2. Compute their cosine similarity
#   3. Compare: what score would BoW give? What score do embeddings give?

# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION

pair_words = ["car", "automobile"]
pair_embeddings = embed_model.encode(pair_words)

sim = cosine_similarity([pair_embeddings[0]], [pair_embeddings[1]])[0][0]

print(f"Similarity between 'car' and 'automobile': {sim:.3f}")
print()
print("BoW similarity:       0.000  (completely different words = no overlap)")
print(f"Embedding similarity: {sim:.3f}  (embeddings understand they mean the same thing!)")
print()
print("💡 This is the fundamental advantage of embeddings over BoW.")
print("   Meaning is captured, not just surface-level string matching.")

In [ ]:
# 🏋️ EXERCISE 4 — Semantic Search (Mini Version)
# ─────────────────────────────────────────────────────────────────────────────
# This is a simplified version of what Chapter 8 covers in depth.
# Semantic search = find documents by MEANING, not exact keywords.
#
# Scenario: You have a help center with these articles:
docs = [
    "How to reset your password",
    "Billing and payment options",
    "Getting started with the mobile app",
    "How to cancel your subscription",
    "Troubleshooting login issues",
    "Upgrading your account plan",
]
#
# User searches: "I forgot my login credentials"
# This does NOT match any doc title exactly — but semantically it's close to
# "How to reset your password" and "Troubleshooting login issues"
#
# Tasks:
#   1. Encode all docs AND the user query using embed_model
#   2. Compute similarity between the query and each doc
#   3. Print docs ranked by similarity (most relevant first)

user_query = "I forgot my login credentials"

# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION

# Encode everything
doc_embeddings = embed_model.encode(docs)
query_embedding = embed_model.encode([user_query])

# Compute similarities
similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

# Rank by similarity
ranked = sorted(zip(similarities, docs), reverse=True)

print(f"Query: '{user_query}'")
print()
print("Results ranked by semantic similarity:")
print("-" * 60)
for rank, (score, doc) in enumerate(ranked, 1):
    bar = "█" * int(score * 30)
    print(f"#{rank}  {score:.3f}  {bar}")
    print(f"     {doc}")
    print()

print("💡 Notice: the query doesn't contain 'reset', 'password', or 'login'")
print("   but the most relevant docs still rank at the top.")
print("   This is the power of semantic search over keyword search.")
print("   Chapter 8 builds a full RAG system using this principle.")

---
## Part 3 — Your First Text Generation

### What we're doing
Now we switch from **representation models** (understanding/embedding text) to **generative models** (producing text). This is the GPT side of things — decoder-only architecture that predicts the next token.

> ⚠️ **GPU Note:** This section works best on **Google Colab** (free T4 GPU).
> Running on CPU is possible but will be slow. The model download is ~2.3 GB.

In [ ]:
# ── DEMO: Load a Generative Model ────────────────────────────────────────────
# Phi-3-mini: 3.8B parameters, MIT licensed, runs on <8GB VRAM
# 'instruct' variant = fine-tuned to follow instructions (not just complete text)

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

model_id = "microsoft/Phi-3-mini-4k-instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model (first run downloads ~2.3 GB — be patient)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",        # GPU if available, else CPU
    torch_dtype="auto",       # Use model's recommended precision
    trust_remote_code=True,   # Phi-3 has custom code in its repo
)

# Create the pipeline — bundles model + tokenizer + generation logic
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,  # Return only the response, not the prompt
    max_new_tokens=300,      # Max length of generated response
    do_sample=False,         # Deterministic — same prompt = same output
                             # Set True for more varied/creative responses
)

print("\n✅ Generative model ready!")

In [ ]:
# ── Helper function for cleaner output ───────────────────────────────────────

def ask(prompt, system="You are a helpful assistant."):
    """Send a single message and print the response."""
    messages = [
        {"role": "system",  "content": system},
        {"role": "user",    "content": prompt}
    ]
    output = generator(messages)
    response = output[0]["generated_text"].strip()
    print(f"👤 You: {prompt}")
    print(f"🤖 LLM: {response}")
    print()
    return response

# The book's first example
ask("Create a funny joke about chickens.")

In [ ]:
# 🏋️ EXERCISE 5 — Explore Prompts
# ─────────────────────────────────────────────────────────────────────────────
# Try these 4 prompts and observe the outputs carefully.
# After each, think: does the response make sense? Is anything wrong?

# 1. Factual question
ask("What is the Transformer architecture in machine learning? Explain in 3 sentences.")

In [ ]:
# 2. Code generation — directly relevant to your work as a fullstack dev
ask("Write a JavaScript function that takes an array of objects and removes duplicates based on an 'id' field.")

In [ ]:
# 3. System prompt matters — compare these two
ask(
    prompt="Explain what an API is.",
    system="You are a teacher explaining concepts to a 10-year-old child."
)

In [ ]:
ask(
    prompt="Explain what an API is.",
    system="You are a senior software architect giving a technical briefing."
)

# 💡 OBSERVE: Same question, completely different responses based on system prompt.
# This is why system prompts are so important when building AI-powered apps.

In [ ]:
# 4. YOUR OWN PROMPT — ask something relevant to your actual work
# Example ideas:
#   - "What are the pros and cons of using microservices vs monolith architecture?"
#   - "Write a Python function that validates an email address"
#   - "Explain the difference between SQL and NoSQL databases"

# YOUR CODE HERE:
ask("...replace this with your own question...")

---
## Part 4 — Context Window in Action

### What we're demonstrating
The context window is everything the model can "see" at once — your system prompt, conversation history, and current message all count toward it. Here we build a simple multi-turn chat to see how the model maintains context across messages.

In [ ]:
# ── DEMO: Multi-turn conversation ────────────────────────────────────────────
# The model remembers context because we send the FULL history every time
# This is how all LLM APIs work — there's no server-side memory by default

conversation_history = [
    {"role": "system", "content": "You are a helpful assistant. Be concise."}
]

def chat(user_message):
    """Chat with memory — sends full history each time."""
    conversation_history.append({"role": "user", "content": user_message})
    
    output = generator(conversation_history)
    assistant_reply = output[0]["generated_text"].strip()
    
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    
    print(f"👤 You: {user_message}")
    print(f"🤖 LLM: {assistant_reply}")
    print(f"   [Context window used: ~{sum(len(m['content']) for m in conversation_history)} chars]")
    print()

# Start a conversation
chat("My name is Alex and I'm learning about LLMs.")
chat("What should I learn after understanding the basics?")
chat("What was my name again?")  # Does it remember?

In [ ]:
# 🏋️ EXERCISE 6 — Context Window Limit
# ─────────────────────────────────────────────────────────────────────────────
# Phi-3-mini has a 4,096 token context window (~3,000 words).
# In a real app, you can't keep the full history forever.
#
# Think about this (no code needed — answer as a comment):
#
#   You're building a customer support chatbot. After 20+ messages,
#   the conversation history might exceed the context window.
#
#   What are THREE strategies you could use to handle this?
#   Write your answers below as comments:

# Strategy 1:

# Strategy 2:

# Strategy 3:


In [ ]:
# ✅ SOLUTION — common strategies for context window management

strategies = """
Strategy 1: SLIDING WINDOW
  Keep only the last N messages in history.
  Simple but loses early context (e.g. user's name, initial problem description).

Strategy 2: SUMMARIZATION
  When history gets long, use the LLM itself to summarize older messages
  into a compact summary, then replace those messages with the summary.
  Preserves key information while reducing token count.

Strategy 3: RAG INSTEAD OF STUFFING
  Instead of putting everything in context, store information externally
  (database, vector store) and retrieve only the relevant parts at query time.
  Most scalable approach — covered in Chapter 8.

Strategy 4 (bonus): ALWAYS KEEP SYSTEM PROMPT + FIRST MESSAGE
  The system prompt and first message often contain critical context
  (user name, subscription tier, preferences). Keep these fixed and
  compress/slide the middle of the conversation.
"""
print(strategies)

---
## Part 5 — Chapter 1 Knowledge Check

**Instructions:** Answer each question in the cell below it.
Write your answers as strings — no looking at notes, try from memory first.
Then run the solution cell to see the model answers.

In [ ]:
# ❓ Q1: What is the core difference between a Representation Model and a
#        Generative Model? Give one real-world example of each.

your_answer_q1 = """
Write your answer here...
"""
print(your_answer_q1)

In [ ]:
# ❓ Q2: What specific problem did the Transformer architecture solve
#        that RNNs with attention could not?

your_answer_q2 = """
Write your answer here...
"""
print(your_answer_q2)

In [ ]:
# ❓ Q3: PRACTICAL SCENARIO
#   You're building a feature for your app where users type a natural language
#   question like "how do I change my email address?" and your app finds the
#   most relevant help article — even if it's titled "updating account details".
#
#   What type of model do you use (representation or generative) and why?
#   Which specific chapter of this book covers this?

your_answer_q3 = """
Write your answer here...
"""
print(your_answer_q3)

In [ ]:
# ❓ Q4: PRACTICAL SCENARIO
#   Your company is building an AI assistant for a healthcare provider.
#   Patients will ask questions and the assistant will respond.
#   Should you use OpenAI's API or host an open model yourself?
#   List at least 2 reasons for your choice.

your_answer_q4 = """
Write your answer here...
"""
print(your_answer_q4)

In [ ]:
# ❓ Q5: What is hallucination in LLMs and why is it dangerous in production apps?
#        Name one technique that helps reduce (but not eliminate) it.

your_answer_q5 = """
Write your answer here...
"""
print(your_answer_q5)

In [ ]:
# ✅ MODEL ANSWERS — run this only after writing your own answers above

model_answers = {
    "Q1": """Representation models (encoder-only, e.g. BERT) process text and produce
embeddings — dense vectors that capture meaning. They don't generate text.
Used for: classification, search, clustering.

Generative models (decoder-only, e.g. GPT-4, Phi-3) process text and produce
more text — they predict the next token repeatedly until the response is complete.
Used for: chat, summarization, code generation, writing.""",

    "Q2": """RNNs processed tokens SEQUENTIALLY — one at a time. This meant training
couldn't be parallelized on GPUs, making it slow and hard to scale.
The Transformer uses self-attention to process ALL tokens in a sequence
SIMULTANEOUSLY, enabling full GPU parallelization and dramatically faster training.
It also handles long-range dependencies better since any two tokens can
directly attend to each other regardless of distance.""",

    "Q3": """Representation model (encoder-only).
This is semantic search — you need to understand the MEANING of both the
query and the help articles, then find the closest match.
Process: encode all help articles as embeddings → encode the user query
→ find the article with the highest cosine similarity to the query.
Covered in Chapter 6 (prompt engineering/search) and Chapter 8 (RAG).""",

    "Q4": """Host an open model yourself. Reasons:
1. PATIENT DATA PRIVACY — healthcare data is highly sensitive and regulated
   (HIPAA in the US). Sending it to OpenAI's API means it leaves your
   infrastructure, creating compliance and liability risks.
2. REGULATORY COMPLIANCE — medical AI applications face strict regulations.
   A self-hosted model gives you full control, auditability, and the ability
   to demonstrate data handling compliance to regulators.
3. Fine-tuning control — you can fine-tune a local model on medical terminology
   and your specific use case, which you can't do with the OpenAI API.""",

    "Q5": """Hallucination is when an LLM confidently generates factually incorrect
information. The model doesn't distinguish between what it knows and what
it's making up — it just predicts the most probable next token.

Dangerous in production because users trust confident-sounding responses.
In medical, legal, or financial contexts this could cause real harm.

RAG (Retrieval-Augmented Generation) helps reduce it by grounding the model's
responses in real retrieved documents rather than relying on memorized training
data. The model is told to only answer based on the provided context.
It doesn't eliminate hallucination but significantly reduces it."""
}

for q, answer in model_answers.items():
    print(f"{'='*65}")
    print(f"✅ {q} — Model Answer:")
    print(answer)
    print()

---
## 🎉 Chapter 1 Practice Complete!

### What you did in this notebook:
- ✅ Built Bag-of-Words from scratch and proved its two core limitations
- ✅ Loaded a real embedding model and measured semantic similarity
- ✅ Built a mini semantic search engine from scratch
- ✅ Loaded and ran a real generative LLM (Phi-3-mini)
- ✅ Experimented with system prompts and saw how they shape responses
- ✅ Built a multi-turn conversation with memory
- ✅ Thought through context window management strategies
- ✅ Answered all knowledge check questions

### Before moving to Chapter 2:
Read `summary.md` once — it's a 10-minute read that locks in everything from this chapter.

### What's coming in Chapter 2:
**Tokens and Embeddings** — we go deep on how text is actually split into tokens (it's not just spaces), what subword tokenization is, and how modern models create context-aware embeddings. The foundation you built here makes Chapter 2 much easier to follow.